# Trabalho de Álgebra Linear em Python

Notebook organizado para **apresentação no Google Colab**, preservando o código original do projeto.

## Estrutura da apresentação
1. Botão para executar tudo
2. Visão geral do projeto
3. Código-fonte separado por arquivos
4. Demonstração principal (`main.py`)
5. Validação por testes (`tests.py`)

> **Observação:** o conteúdo dos arquivos foi mantido, apenas a organização visual do notebook foi melhorada.


## Execução rápida
Execute a célula abaixo para tentar rodar o notebook inteiro automaticamente no Colab.

In [ ]:
from IPython.display import Javascript, display

display(Javascript('IPython.notebook.execute_all_cells()'))
print("Comando enviado: executar todas as células do notebook.")


## Visão geral do projeto

Este trabalho implementa, em **Python puro**, estruturas e operações de Álgebra Linear sem uso de bibliotecas externas como `numpy`.

### Arquivos principais
- `matrix.py`: classe `Matrix`
- `vector.py`: classe `Vector`
- `linear_algebra.py`: operações e algoritmos
- `main.py`: demonstração prática
- `tests.py`: testes automatizados


## Documentação do projeto (`README.md`)

# Trabalho de Algebra Linear em Python

## Objetivo do trabalho

Este trabalho tem como objetivo implementar, em Python puro e sem o uso de bibliotecas externas como `numpy`, estruturas basicas de Algebra Linear para representar matrizes e vetores, realizar operacoes elementares e resolver sistemas lineares por eliminacao de Gauss e Gauss-Jordan.

## Classes implementadas

- `Matrix`
  Representa uma matriz numerica, com validacao de dimensoes, acesso por indices, copia independente e alguns atalhos de operacoes.
- `Vector`
  Representa um vetor numerico, com validacao de dimensao, acesso por indice, copia independente e conversao para matriz coluna.
- `LinearAlgebra`
  Reune os algoritmos e operacoes algebricas entre matrizes e vetores.

## Metodos disponiveis

### `Matrix`

- `__init__(rows, cols, elements)`
- `get(i, j)` e `set(i, j, value)`
- `shape()`
- `copy()` e `to_list()`
- `transpose()`
- `add(other)` e `subtract(other)`
- `scalar_multiply(scalar)`
- `is_square()`
- `swap_rows(row_a, row_b)`
- `multiply_row(row, scalar)`
- `add_multiple_of_row(source_row, target_row, scalar)`
- `gaussian_elimination()`
- `gauss_jordan_elimination()`

### `Vector`

- `__init__(dim, elements)`
- `get(i)` e `set(i, value)`
- `copy()` e `to_list()`
- `add(other)` e `subtract(other)`
- `scalar_multiply(scalar)`
- `dot(other)`
- `to_column_matrix()`

### `LinearAlgebra`

- `transpose(a)`
- `sum(a, b)`
- `times(a, b)`
- `dot(a, b)`
- `gauss(a)`
- `gauss_jordan(a)`
- `solve(a)`
- `create_identity(size)`
- `augment_matrix(matrix, vector)`
- `solve_system(matrix, vector)`

## Exemplos de uso

```python
from matrix import Matrix
from vector import Vector
from linear_algebra import LinearAlgebra

matrix = Matrix(2, 2, [1, 2, 3, 4])
vector = Vector(2, [5, 6])

print(matrix)
print(vector)

editable_matrix = matrix.copy()
print(editable_matrix.get(0, 1))
editable_matrix.set(0, 1, 20)
print(editable_matrix)

print(matrix.transpose())
print(LinearAlgebra.sum(matrix, Matrix(2, 2, [10, 20, 30, 40])))
print(LinearAlgebra.times(2, vector))

matrix_a = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
matrix_b = Matrix(3, 2, [7, 8, 9, 10, 11, 12])
print(LinearAlgebra.dot(matrix_a, matrix_b))

gauss_matrix = Matrix(3, 3, [2, 1, -1, -3, -1, 2, -2, 1, 2])
print(LinearAlgebra.gauss(gauss_matrix))

system = Matrix(2, 3, [1, 1, 3, 2, -1, 0])
print(LinearAlgebra.solve(system))
```

## Como executar os testes

No terminal, execute:

```bash
python tests.py
```

Em algumas instalacoes do Windows, pode ser necessario usar:

```bash
py tests.py
```

## Decisoes de implementacao

- Os dados internos sao armazenados com listas nativas do Python.
- `Matrix` e `Vector` concentram a representacao e as validacoes basicas.
- `LinearAlgebra` concentra os algoritmos e operacoes entre objetos.
- Os algoritmos de eliminacao trabalham sobre copias, preservando os dados originais.
- O metodo `solve(a)` recebe uma matriz aumentada e retorna a forma reduzida final quando o sistema possui solucao unica.

## Tratamento de erros adotado

- `TypeError` e usado quando o tipo do argumento nao e valido.
- `ValueError` e usado quando a dimensao ou a estrutura dos dados nao e compativel com a operacao.
- `IndexError` e usado quando um indice de linha, coluna ou posicao esta fora do intervalo permitido.
- Em sistemas lineares, `solve(a)` gera `ValueError` para sistema impossivel ou com multiplas solucoes.


## Código-fonte: `matrix.py`
Classe responsável pela representação e manipulação de matrizes.

In [ ]:
class Matrix:
    """Representa uma matriz numerica usando listas nativas do Python."""

    def __init__(self, rows, cols, elements):
        self._validate_dimension("rows", rows)
        self._validate_dimension("cols", cols)

        normalized_elements = self._normalize_elements(elements)
        expected_size = rows * cols

        if len(normalized_elements) != expected_size:
            raise ValueError(
                "elements deve conter exatamente "
                f"{expected_size} valores, mas recebeu {len(normalized_elements)}."
            )

        self.rows = rows
        self.cols = cols
        self.data = self._build_data(normalized_elements)

    def __repr__(self):
        return (
            f"Matrix(rows={self.rows}, cols={self.cols}, "
            f"elements={self._flatten_elements()})"
        )

    def __str__(self):
        lines = []
        for row in self.data:
            row_text = " ".join(str(value) for value in row)
            lines.append(f"[ {row_text} ]")
        return "\n".join(lines)

    @staticmethod
    def _validate_dimension(name, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} deve ser um inteiro positivo.")

        if value <= 0:
            raise ValueError(f"{name} deve ser maior que zero.")

    @staticmethod
    def _normalize_elements(elements):
        if not isinstance(elements, (list, tuple)):
            raise TypeError(
                "elements deve ser uma lista ou tupla de valores numericos."
            )

        normalized_elements = list(elements)

        for index, value in enumerate(normalized_elements):
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError(
                    f"Elemento invalido na posicao {index}: o valor deve ser numerico."
                )

        return normalized_elements

    def _build_data(self, elements):
        data = []

        for row_index in range(self.rows):
            start = row_index * self.cols
            end = start + self.cols
            data.append(elements[start:end])

        return data

    def _flatten_elements(self):
        return [value for row in self.data for value in row]

    def _validate_indices(self, i, j):
        if isinstance(i, bool) or not isinstance(i, int):
            raise TypeError("O indice da linha deve ser um inteiro.")

        if isinstance(j, bool) or not isinstance(j, int):
            raise TypeError("O indice da coluna deve ser um inteiro.")

        if i < 0 or i >= self.rows:
            raise IndexError(
                "Indice de linha fora do intervalo: esperado entre 0 e "
                f"{self.rows - 1}, mas recebeu {i}."
            )

        if j < 0 or j >= self.cols:
            raise IndexError(
                "Indice de coluna fora do intervalo: esperado entre 0 e "
                f"{self.cols - 1}, mas recebeu {j}."
            )

    @staticmethod
    def _validate_value(value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("O valor da matriz deve ser numerico.")

    def shape(self):
        """Retorna a dimensao da matriz."""
        return self.rows, self.cols

    def copy(self):
        """Retorna uma copia independente da matriz."""
        return Matrix(self.rows, self.cols, self._flatten_elements())

    def to_list(self):
        """Retorna uma copia dos dados internos da matriz."""
        return [row[:] for row in self.data]

    def get(self, i, j):
        """Retorna o elemento na posicao informada."""
        self._validate_indices(i, j)
        return self.data[i][j]

    def set(self, i, j, value):
        """Atualiza o elemento na posicao informada."""
        self._validate_indices(i, j)
        self._validate_value(value)
        self.data[i][j] = value

    def transpose(self):
        """Retorna a transposta da matriz."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.transpose(self)

    def add(self, other):
        """Soma duas matrizes de mesma ordem."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.sum(self, other)

    def subtract(self, other):
        """Subtrai duas matrizes de mesma ordem."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.sum(self, LinearAlgebra.times(-1, other))

    def scalar_multiply(self, scalar):
        """Multiplica a matriz por um escalar."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.times(scalar, self)

    def multiply(self, other):
        """Multiplica a matriz por outra matriz, vetor ou escalar."""
        raise NotImplementedError(
            "Metodo multiply nao faz parte do escopo principal do trabalho. "
            "Use LinearAlgebra.times ou LinearAlgebra.dot conforme a operacao desejada."
        )

    def is_square(self):
        """Verifica se a matriz e quadrada."""
        return self.rows == self.cols

    def swap_rows(self, row_a, row_b):
        """Troca duas linhas da matriz."""
        self._validate_indices(row_a, 0)
        self._validate_indices(row_b, 0)

        if row_a == row_b:
            return

        self.data[row_a], self.data[row_b] = self.data[row_b], self.data[row_a]

    def multiply_row(self, row, scalar):
        """Multiplica uma linha por um escalar."""
        self._validate_indices(row, 0)
        self._validate_value(scalar)

        for col in range(self.cols):
            self.data[row][col] *= scalar

    def add_multiple_of_row(self, source_row, target_row, scalar):
        """Soma um multiplo de uma linha em outra linha."""
        self._validate_indices(source_row, 0)
        self._validate_indices(target_row, 0)
        self._validate_value(scalar)

        for col in range(self.cols):
            self.data[target_row][col] += scalar * self.data[source_row][col]

    def gaussian_elimination(self):
        """Aplica a eliminacao gaussiana e retorna a matriz escalonada."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.gauss(self)

    def gauss_jordan_elimination(self):
        """Aplica Gauss-Jordan e retorna a matriz em forma escalonada reduzida."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.gauss_jordan(self)


## Código-fonte: `vector.py`
Classe responsável pela representação e manipulação de vetores.

In [ ]:
class Vector:
    """Representa um vetor numerico usando listas nativas do Python."""

    def __init__(self, dim, elements):
        self._validate_dimension(dim)

        normalized_elements = self._normalize_elements(elements)

        if len(normalized_elements) != dim:
            raise ValueError(
                "elements deve conter exatamente "
                f"{dim} valores, mas recebeu {len(normalized_elements)}."
            )

        self.dim = dim
        self.size = dim
        self.data = normalized_elements

    def __repr__(self):
        return f"Vector(dim={self.dim}, elements={self.data})"

    def __str__(self):
        values = " ".join(str(value) for value in self.data)
        return f"[ {values} ]"

    @staticmethod
    def _validate_dimension(dim):
        if isinstance(dim, bool) or not isinstance(dim, int):
            raise TypeError("dim deve ser um inteiro positivo.")

        if dim <= 0:
            raise ValueError("dim deve ser maior que zero.")

    @staticmethod
    def _normalize_elements(elements):
        if not isinstance(elements, (list, tuple)):
            raise TypeError(
                "elements deve ser uma lista ou tupla de valores numericos."
            )

        normalized_elements = list(elements)

        for index, value in enumerate(normalized_elements):
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError(
                    f"Elemento invalido na posicao {index}: o valor deve ser numerico."
                )

        return normalized_elements

    def _validate_index(self, i):
        if isinstance(i, bool) or not isinstance(i, int):
            raise TypeError("O indice do vetor deve ser um inteiro.")

        if i < 0 or i >= self.dim:
            raise IndexError(
                "Indice fora do intervalo: esperado entre 0 e "
                f"{self.dim - 1}, mas recebeu {i}."
            )

    @staticmethod
    def _validate_value(value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("O valor do vetor deve ser numerico.")

    def __len__(self):
        return self.dim

    def copy(self):
        """Retorna uma copia independente do vetor."""
        return Vector(self.dim, self.to_list())

    def to_list(self):
        """Retorna uma copia dos dados internos do vetor."""
        return self.data[:]

    def get(self, i):
        """Retorna o elemento na posicao informada."""
        self._validate_index(i)
        return self.data[i]

    def set(self, i, value):
        """Atualiza o elemento na posicao informada."""
        self._validate_index(i)
        self._validate_value(value)
        self.data[i] = value

    def add(self, other):
        """Soma dois vetores de mesmo tamanho."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.sum(self, other)

    def subtract(self, other):
        """Subtrai dois vetores de mesmo tamanho."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.sum(self, LinearAlgebra.times(-1, other))

    def scalar_multiply(self, scalar):
        """Multiplica o vetor por um escalar."""
        from linear_algebra import LinearAlgebra

        return LinearAlgebra.times(scalar, self)

    def dot(self, other):
        """Calcula o produto escalar entre dois vetores."""
        if not isinstance(other, Vector):
            raise TypeError("dot aceita apenas outro objeto Vector.")

        if self.dim != other.dim:
            raise ValueError("Os vetores devem possuir a mesma dimensao.")

        total = 0

        for i in range(self.dim):
            total += self.get(i) * other.get(i)

        return total

    def to_column_matrix(self):
        """Converte o vetor para uma matriz coluna."""
        from matrix import Matrix

        return Matrix(self.dim, 1, self.to_list())


## Código-fonte: `linear_algebra.py`
Classe com operações e algoritmos de Álgebra Linear.

In [ ]:
from matrix import Matrix
from vector import Vector


class LinearAlgebra:
    """Agrupa operacoes e algoritmos de Algebra Linear."""

    EPSILON = 1e-10

    @staticmethod
    def validate_same_matrix_order(matrix_a, matrix_b):
        """Valida se duas matrizes possuem a mesma ordem."""
        if not isinstance(matrix_a, Matrix) or not isinstance(matrix_b, Matrix):
            raise TypeError("Os dois argumentos devem ser objetos Matrix.")

        if matrix_a.rows != matrix_b.rows or matrix_a.cols != matrix_b.cols:
            raise ValueError("As matrizes devem possuir a mesma dimensao.")

    @staticmethod
    def validate_same_vector_size(vector_a, vector_b):
        """Valida se dois vetores possuem o mesmo tamanho."""
        if not isinstance(vector_a, Vector) or not isinstance(vector_b, Vector):
            raise TypeError("Os dois argumentos devem ser objetos Vector.")

        if vector_a.dim != vector_b.dim:
            raise ValueError("Os vetores devem possuir a mesma dimensao.")

    @staticmethod
    def validate_matrix_vector_dimensions(matrix, vector):
        """Valida se matriz e vetor sao compativeis em dimensao."""
        if not isinstance(matrix, Matrix):
            raise TypeError("O primeiro argumento deve ser um objeto Matrix.")

        if not isinstance(vector, Vector):
            raise TypeError("O segundo argumento deve ser um objeto Vector.")

        if matrix.cols != vector.dim:
            raise ValueError(
                "O numero de colunas da matriz deve ser igual a dimensao do vetor."
            )

    @staticmethod
    def _validate_system_vector_dimensions(matrix, vector):
        if not isinstance(matrix, Matrix):
            raise TypeError("O primeiro argumento deve ser um objeto Matrix.")

        if not isinstance(vector, Vector):
            raise TypeError("O segundo argumento deve ser um objeto Vector.")

        if matrix.rows != vector.dim:
            raise ValueError(
                "A dimensao do vetor deve ser igual ao numero de linhas da matriz."
            )

    @staticmethod
    def _validate_positive_integer(name, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} deve ser um inteiro positivo.")

        if value <= 0:
            raise ValueError(f"{name} deve ser maior que zero.")

    @staticmethod
    def _is_scalar(value):
        return isinstance(value, (int, float)) and not isinstance(value, bool)

    @staticmethod
    def _map_matrix(matrix, operation):
        elements = []

        for i in range(matrix.rows):
            for j in range(matrix.cols):
                elements.append(operation(matrix.get(i, j)))

        return Matrix(matrix.rows, matrix.cols, elements)

    @staticmethod
    def _map_vector(vector, operation):
        elements = []

        for i in range(vector.dim):
            elements.append(operation(vector.get(i)))

        return Vector(vector.dim, elements)

    @staticmethod
    def _combine_matrices(matrix_a, matrix_b, operation):
        elements = []

        for i in range(matrix_a.rows):
            for j in range(matrix_a.cols):
                elements.append(operation(matrix_a.get(i, j), matrix_b.get(i, j)))

        return Matrix(matrix_a.rows, matrix_a.cols, elements)

    @staticmethod
    def _combine_vectors(vector_a, vector_b, operation):
        elements = []

        for i in range(vector_a.dim):
            elements.append(operation(vector_a.get(i), vector_b.get(i)))

        return Vector(vector_a.dim, elements)

    @staticmethod
    def _validate_matrix_argument(matrix, method_name):
        if not isinstance(matrix, Matrix):
            raise TypeError(f"{method_name} aceita apenas objetos Matrix.")

    @staticmethod
    def _validate_augmented_matrix(matrix, method_name):
        LinearAlgebra._validate_matrix_argument(matrix, method_name)

        if matrix.cols < 2:
            raise ValueError(
                "A matriz aumentada deve ter pelo menos duas colunas."
            )

    @staticmethod
    def _run_elimination(matrix, coefficient_columns, reduced):
        pivot_row = 0

        for pivot_col in range(coefficient_columns):
            if pivot_row >= matrix.rows:
                break

            best_row = LinearAlgebra._find_pivot_row(matrix, pivot_row, pivot_col)

            if best_row is None:
                continue

            matrix.swap_rows(pivot_row, best_row)

            if reduced:
                LinearAlgebra._normalize_pivot_row(matrix, pivot_row, pivot_col)
                LinearAlgebra._eliminate_other_rows(matrix, pivot_row, pivot_col)
            else:
                LinearAlgebra._eliminate_below_pivot(matrix, pivot_row, pivot_col)

            pivot_row += 1

        LinearAlgebra._cleanup_near_zero(matrix)

    @staticmethod
    def _find_pivot_row(matrix, start_row, col):
        pivot_row = start_row
        pivot_value = abs(matrix.get(start_row, col))

        for row in range(start_row + 1, matrix.rows):
            current_value = abs(matrix.get(row, col))

            if current_value > pivot_value:
                pivot_value = current_value
                pivot_row = row

        if pivot_value <= LinearAlgebra.EPSILON:
            return None

        return pivot_row

    @staticmethod
    def _eliminate_below_pivot(matrix, pivot_row, pivot_col):
        pivot_value = matrix.get(pivot_row, pivot_col)

        if abs(pivot_value) <= LinearAlgebra.EPSILON:
            return

        for row in range(pivot_row + 1, matrix.rows):
            current_value = matrix.get(row, pivot_col)

            if abs(current_value) <= LinearAlgebra.EPSILON:
                matrix.set(row, pivot_col, 0.0)
                continue

            factor = current_value / pivot_value
            matrix.add_multiple_of_row(pivot_row, row, -factor)

    @staticmethod
    def _cleanup_near_zero(matrix):
        for row in range(matrix.rows):
            for col in range(matrix.cols):
                value = matrix.get(row, col)

                if abs(value) <= LinearAlgebra.EPSILON:
                    matrix.set(row, col, 0.0)

    @staticmethod
    def _normalize_pivot_row(matrix, pivot_row, pivot_col):
        pivot_value = matrix.get(pivot_row, pivot_col)

        if abs(pivot_value) <= LinearAlgebra.EPSILON:
            raise ValueError("Nao foi possivel normalizar a linha do pivo.")

        matrix.multiply_row(pivot_row, 1 / pivot_value)

    @staticmethod
    def _eliminate_other_rows(matrix, pivot_row, pivot_col):
        for row in range(matrix.rows):
            if row == pivot_row:
                continue

            factor = matrix.get(row, pivot_col)

            if abs(factor) <= LinearAlgebra.EPSILON:
                matrix.set(row, pivot_col, 0.0)
                continue

            matrix.add_multiple_of_row(pivot_row, row, -factor)

    @staticmethod
    def _row_has_non_zero_coefficients(matrix, row):
        last_col = matrix.cols - 1

        for col in range(last_col):
            if abs(matrix.get(row, col)) > LinearAlgebra.EPSILON:
                return True

        return False

    @staticmethod
    def _is_inconsistent_row(matrix, row):
        if LinearAlgebra._row_has_non_zero_coefficients(matrix, row):
            return False

        return abs(matrix.get(row, matrix.cols - 1)) > LinearAlgebra.EPSILON

    @staticmethod
    def _leading_coefficient_column(matrix, row):
        last_col = matrix.cols - 1

        for col in range(last_col):
            if abs(matrix.get(row, col)) > LinearAlgebra.EPSILON:
                return col

        return None

    @staticmethod
    def _count_pivots_in_coefficients(matrix):
        pivot_count = 0

        for row in range(matrix.rows):
            leading_col = LinearAlgebra._leading_coefficient_column(matrix, row)

            if leading_col is None:
                continue

            if abs(matrix.get(row, leading_col) - 1.0) <= LinearAlgebra.EPSILON:
                pivot_count += 1

        return pivot_count

    @staticmethod
    def transpose(a):
        """Retorna a transposta de uma matriz ou vetor."""
        if isinstance(a, Matrix):
            elements = []

            for j in range(a.cols):
                for i in range(a.rows):
                    elements.append(a.get(i, j))

            return Matrix(a.cols, a.rows, elements)

        if isinstance(a, Vector):
            return Matrix(1, a.dim, a.to_list())

        raise TypeError("transpose aceita apenas objetos Matrix ou Vector.")

    @staticmethod
    def sum(a, b):
        """Soma matrizes ou vetores de mesma dimensao."""
        if isinstance(a, Matrix) and isinstance(b, Matrix):
            LinearAlgebra.validate_same_matrix_order(a, b)
            return LinearAlgebra._combine_matrices(a, b, lambda x, y: x + y)

        if isinstance(a, Vector) and isinstance(b, Vector):
            LinearAlgebra.validate_same_vector_size(a, b)
            return LinearAlgebra._combine_vectors(a, b, lambda x, y: x + y)

        raise TypeError(
            "sum aceita apenas Matrix com Matrix ou Vector com Vector."
        )

    @staticmethod
    def times(a, b):
        """Realiza multiplicacao por escalar ou elemento a elemento."""
        if LinearAlgebra._is_scalar(a) and isinstance(b, Matrix):
            return LinearAlgebra._map_matrix(b, lambda value: a * value)

        if isinstance(a, Matrix) and LinearAlgebra._is_scalar(b):
            return LinearAlgebra._map_matrix(a, lambda value: value * b)

        if LinearAlgebra._is_scalar(a) and isinstance(b, Vector):
            return LinearAlgebra._map_vector(b, lambda value: a * value)

        if isinstance(a, Vector) and LinearAlgebra._is_scalar(b):
            return LinearAlgebra._map_vector(a, lambda value: value * b)

        if isinstance(a, Matrix) and isinstance(b, Matrix):
            LinearAlgebra.validate_same_matrix_order(a, b)
            return LinearAlgebra._combine_matrices(a, b, lambda x, y: x * y)

        if isinstance(a, Vector) and isinstance(b, Vector):
            LinearAlgebra.validate_same_vector_size(a, b)
            return LinearAlgebra._combine_vectors(a, b, lambda x, y: x * y)

        raise TypeError(
            "times aceita apenas escalar com Matrix, escalar com Vector, "
            "Matrix com Matrix ou Vector com Vector."
        )

    @staticmethod
    def dot(a, b):
        """Realiza a multiplicacao matricial entre duas matrizes."""
        if not isinstance(a, Matrix) or not isinstance(b, Matrix):
            raise TypeError("dot aceita apenas objetos Matrix.")

        if a.cols != b.rows:
            raise ValueError(
                "Para multiplicacao matricial, o numero de colunas da primeira "
                "matriz deve ser igual ao numero de linhas da segunda."
            )

        elements = []

        for i in range(a.rows):
            for j in range(b.cols):
                total = 0

                for k in range(a.cols):
                    total += a.get(i, k) * b.get(k, j)

                elements.append(total)

        return Matrix(a.rows, b.cols, elements)

    @staticmethod
    def gauss(a):
        """Aplica eliminacao gaussiana e retorna uma nova matriz escalonada."""
        LinearAlgebra._validate_matrix_argument(a, "gauss")

        result = a.copy()
        LinearAlgebra._run_elimination(result, result.cols, reduced=False)
        return result

    @staticmethod
    def gauss_jordan(a):
        """Aplica Gauss-Jordan e retorna uma nova matriz reduzida por linhas."""
        LinearAlgebra._validate_matrix_argument(a, "gauss_jordan")

        result = a.copy()
        LinearAlgebra._run_elimination(result, result.cols, reduced=True)
        return result

    @staticmethod
    def solve(a):
        """
        Resolve um sistema linear dado por uma matriz aumentada.

        O retorno e a propria matriz aumentada em forma reduzida por linhas
        somente quando o sistema possui solucao unica. Se o sistema for
        impossivel ou possuir multiplas solucoes, o metodo gera ValueError.
        """
        LinearAlgebra._validate_augmented_matrix(a, "solve")

        result = a.copy()
        variable_count = result.cols - 1
        LinearAlgebra._run_elimination(result, variable_count, reduced=True)

        for row in range(result.rows):
            if LinearAlgebra._is_inconsistent_row(result, row):
                raise ValueError("O sistema e impossivel: nao possui solucao.")

        pivot_count = LinearAlgebra._count_pivots_in_coefficients(result)

        if pivot_count < variable_count:
            raise ValueError(
                "O sistema possui multiplas solucoes: existem variaveis livres."
            )

        return result

    @staticmethod
    def create_identity(size):
        """Cria uma matriz identidade."""
        LinearAlgebra._validate_positive_integer("size", size)

        elements = []

        for i in range(size):
            for j in range(size):
                elements.append(1 if i == j else 0)

        return Matrix(size, size, elements)

    @staticmethod
    def augment_matrix(matrix, vector):
        """Monta a matriz aumentada [A|b]."""
        LinearAlgebra._validate_system_vector_dimensions(matrix, vector)

        elements = []

        for i in range(matrix.rows):
            for j in range(matrix.cols):
                elements.append(matrix.get(i, j))

            elements.append(vector.get(i))

        return Matrix(matrix.rows, matrix.cols + 1, elements)

    @staticmethod
    def gaussian_elimination(matrix):
        """Executa a eliminacao gaussiana sobre uma matriz."""
        return LinearAlgebra.gauss(matrix)

    @staticmethod
    def solve_system(matrix, vector):
        """Resolve um sistema linear usando Gauss-Jordan."""
        augmented_matrix = LinearAlgebra.augment_matrix(matrix, vector)
        return LinearAlgebra.solve(augmented_matrix)


## Código-fonte: `main.py`
Arquivo de demonstração principal do projeto.

In [ ]:
from linear_algebra import LinearAlgebra
from matrix import Matrix
from vector import Vector


def print_section(title):
    print(f"=== {title} ===")


def main():
    rows_a = 2
    cols_a = 2
    elements_a = [1, 2, 3, 4]

    dim_v = 2
    elements_v = [5, 6]

    rows_b = 2
    cols_b = 2
    elements_b = [10, 20, 30, 40]

    dot_rows_a = 2
    dot_cols_a = 3
    dot_elements_a = [1, 2, 3, 4, 5, 6]

    dot_rows_b = 3
    dot_cols_b = 2
    dot_elements_b = [7, 8, 9, 10, 11, 12]

    gauss_rows = 3
    gauss_cols = 3
    gauss_elements = [2, 1, -1, -3, -1, 2, -2, 1, 2]

    system_rows = 2
    system_cols = 3
    system_elements = [1, 1, 3, 2, -1, 0]

    matrix_a = Matrix(rows_a, cols_a, elements_a)
    vector_v = Vector(dim_v, elements_v)
    editable_matrix = Matrix(rows_a, cols_a, elements_a)
    matrix_b = Matrix(rows_b, cols_b, elements_b)
    dot_matrix_a = Matrix(dot_rows_a, dot_cols_a, dot_elements_a)
    dot_matrix_b = Matrix(dot_rows_b, dot_cols_b, dot_elements_b)
    gauss_matrix = Matrix(gauss_rows, gauss_cols, gauss_elements)
    system_matrix = Matrix(system_rows, system_cols, system_elements)

    print_section("Criacao de Matriz")
    print(matrix_a)
    print()

    print_section("Criacao de Vetor")
    print(vector_v)
    print()

    print_section("Uso de get e set")
    print("Valor original em (0, 1):", editable_matrix.get(0, 1))
    editable_matrix.set(0, 1, 20)
    print("Matriz apos set(0, 1, 20):")
    print(editable_matrix)
    print()

    print_section("Transposta da Matriz")
    print(matrix_a.transpose())
    print()

    print_section("Soma de Matrizes")
    matrix_sum = LinearAlgebra.sum(matrix_a, matrix_b)
    print(matrix_sum)
    print()

    print_section("Multiplicacao por Escalar")
    print(LinearAlgebra.times(2, vector_v))
    print()

    print_section("Multiplicacao de Matrizes (Produto Matricial)")
    print(LinearAlgebra.dot(dot_matrix_a, dot_matrix_b))
    print()

    print_section("Eliminacao Gaussiana")
    print(LinearAlgebra.gauss(gauss_matrix))
    print()

    print_section("Resolucao de Sistema Linear (Gauss-Jordan)")
    print(LinearAlgebra.solve(system_matrix))


if __name__ == "__main__":
    main()


## Código-fonte: `tests.py`
Conjunto de testes para validar o funcionamento do sistema.

In [ ]:
from linear_algebra import LinearAlgebra
from matrix import Matrix
from vector import Vector


def assert_close(actual, expected, tolerance=1e-9):
    assert abs(actual - expected) <= tolerance, (
        f"Valor esperado {expected}, mas recebeu {actual}."
    )


def assert_matrix_values(matrix, expected):
    assert matrix.rows == len(expected), (
        f"Numero de linhas esperado {len(expected)}, mas recebeu {matrix.rows}."
    )
    assert matrix.cols == len(expected[0]), (
        f"Numero de colunas esperado {len(expected[0])}, mas recebeu {matrix.cols}."
    )

    for i, row in enumerate(expected):
        for j, value in enumerate(row):
            assert_close(matrix.get(i, j), value)


def assert_vector_values(vector, expected):
    assert vector.dim == len(expected), (
        f"Dimensao esperada {len(expected)}, mas recebeu {vector.dim}."
    )

    for i, value in enumerate(expected):
        assert_close(vector.get(i), value)


def assert_raises(expected_exception, function, *args, message_contains=None):
    try:
        function(*args)
    except expected_exception as error:
        if message_contains is not None:
            assert message_contains in str(error), (
                f"Mensagem esperada contendo '{message_contains}', "
                f"mas recebeu '{error}'."
            )
        return

    raise AssertionError(
        f"Era esperado que {function.__name__} gerasse {expected_exception.__name__}."
    )


def test_matrix_creation_valid():
    matrix = Matrix(2, 3, [1, 2, 3, 4, 5, 6])

    assert isinstance(matrix, Matrix)
    assert matrix.rows == 2
    assert matrix.cols == 3
    assert matrix.shape() == (2, 3)
    assert matrix.is_square() is False
    assert_matrix_values(matrix, [[1, 2, 3], [4, 5, 6]])


def test_matrix_creation_invalid_size():
    assert_raises(ValueError, Matrix, 0, 2, [], message_contains="maior que zero")
    assert_raises(ValueError, Matrix, 2, 0, [], message_contains="maior que zero")


def test_matrix_creation_invalid_number_of_elements():
    assert_raises(ValueError, Matrix, 2, 2, [1, 2, 3], message_contains="exatamente")


def test_matrix_get_valid():
    matrix = Matrix(2, 2, [10, 20, 30, 40])

    assert_close(matrix.get(0, 0), 10)
    assert_close(matrix.get(1, 1), 40)


def test_matrix_get_invalid():
    matrix = Matrix(2, 2, [1, 2, 3, 4])

    assert_raises(
        IndexError,
        matrix.get,
        2,
        0,
        message_contains="Indice de linha fora do intervalo",
    )
    assert_raises(
        IndexError,
        matrix.get,
        0,
        2,
        message_contains="Indice de coluna fora do intervalo",
    )


def test_matrix_set_valid():
    matrix = Matrix(2, 2, [1, 2, 3, 4])

    matrix.set(1, 0, 99)

    assert_close(matrix.get(1, 0), 99)
    assert_matrix_values(matrix, [[1, 2], [99, 4]])


def test_matrix_set_invalid():
    matrix = Matrix(2, 2, [1, 2, 3, 4])

    assert_raises(
        IndexError,
        matrix.set,
        3,
        0,
        10,
        message_contains="Indice de linha fora do intervalo",
    )
    assert_raises(
        IndexError,
        matrix.set,
        0,
        3,
        10,
        message_contains="Indice de coluna fora do intervalo",
    )


def test_matrix_copy_creates_independent_object():
    matrix = Matrix(2, 2, [1, 2, 3, 4])
    matrix_copy = matrix.copy()

    matrix_copy.set(0, 0, 100)

    assert_close(matrix.get(0, 0), 1)
    assert_close(matrix_copy.get(0, 0), 100)


def test_matrix_wrappers():
    matrix_a = Matrix(2, 2, [1, 2, 3, 4])
    matrix_b = Matrix(2, 2, [4, 3, 2, 1])

    assert_matrix_values(matrix_a.transpose(), [[1, 3], [2, 4]])
    assert_matrix_values(matrix_a.add(matrix_b), [[5, 5], [5, 5]])
    assert_matrix_values(matrix_a.subtract(matrix_b), [[-3, -1], [1, 3]])
    assert_matrix_values(matrix_a.scalar_multiply(2), [[2, 4], [6, 8]])


def test_matrix_row_operations():
    matrix = Matrix(2, 2, [1, 2, 3, 4])

    matrix.swap_rows(0, 1)
    assert_matrix_values(matrix, [[3, 4], [1, 2]])

    matrix.multiply_row(0, 2)
    assert_matrix_values(matrix, [[6, 8], [1, 2]])

    matrix.add_multiple_of_row(1, 0, -2)
    assert_matrix_values(matrix, [[4, 4], [1, 2]])


def test_matrix_gauss_jordan_elimination_wrapper():
    matrix = Matrix(2, 2, [1, 2, 3, 4])
    result = matrix.gauss_jordan_elimination()

    assert_matrix_values(result, [[1, 0], [0, 1]])


def test_vector_creation_valid():
    vector = Vector(3, [1, 2, 3])

    assert isinstance(vector, Vector)
    assert vector.dim == 3
    assert len(vector) == 3
    assert_vector_values(vector, [1, 2, 3])


def test_vector_creation_invalid_dimension():
    assert_raises(ValueError, Vector, 0, [], message_contains="maior que zero")


def test_vector_creation_invalid_number_of_elements():
    assert_raises(ValueError, Vector, 3, [1, 2], message_contains="exatamente")


def test_vector_get_valid():
    vector = Vector(3, [10, 20, 30])

    assert_close(vector.get(0), 10)
    assert_close(vector.get(2), 30)


def test_vector_get_invalid():
    vector = Vector(3, [1, 2, 3])

    assert_raises(
        IndexError,
        vector.get,
        3,
        message_contains="Indice fora do intervalo",
    )


def test_vector_set_valid():
    vector = Vector(3, [1, 2, 3])

    vector.set(1, 99)

    assert_vector_values(vector, [1, 99, 3])


def test_vector_set_invalid():
    vector = Vector(3, [1, 2, 3])

    assert_raises(
        IndexError,
        vector.set,
        3,
        10,
        message_contains="Indice fora do intervalo",
    )


def test_vector_copy_creates_independent_object():
    vector = Vector(3, [1, 2, 3])
    vector_copy = vector.copy()

    vector_copy.set(0, 100)

    assert_close(vector.get(0), 1)
    assert_close(vector_copy.get(0), 100)


def test_vector_wrappers_and_dot():
    vector_a = Vector(3, [1, 2, 3])
    vector_b = Vector(3, [4, 5, 6])

    assert_vector_values(vector_a.add(vector_b), [5, 7, 9])
    assert_vector_values(vector_b.subtract(vector_a), [3, 3, 3])
    assert_vector_values(vector_a.scalar_multiply(3), [3, 6, 9])
    assert_close(vector_a.dot(vector_b), 32)


def test_vector_to_column_matrix():
    vector = Vector(3, [1, 2, 3])
    result = vector.to_column_matrix()

    assert_matrix_values(result, [[1], [2], [3]])


def test_linear_algebra_transpose_matrix():
    matrix = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    result = LinearAlgebra.transpose(matrix)

    assert_matrix_values(result, [[1, 4], [2, 5], [3, 6]])


def test_linear_algebra_transpose_matrix_3x2():
    matrix = Matrix(3, 2, [1, 2, 3, 4, 5, 6])
    result = LinearAlgebra.transpose(matrix)

    assert_matrix_values(result, [[1, 3, 5], [2, 4, 6]])


def test_linear_algebra_transpose_vector():
    vector = Vector(3, [7, 8, 9])
    result = LinearAlgebra.transpose(vector)

    assert isinstance(result, Matrix)
    assert_matrix_values(result, [[7, 8, 9]])


def test_linear_algebra_sum_matrix_valid():
    matrix_a = Matrix(2, 2, [1, 2, 3, 4])
    matrix_b = Matrix(2, 2, [10, 20, 30, 40])
    result = LinearAlgebra.sum(matrix_a, matrix_b)

    assert_matrix_values(result, [[11, 22], [33, 44]])


def test_linear_algebra_sum_vector_valid():
    vector_a = Vector(3, [1, 2, 3])
    vector_b = Vector(3, [10, 20, 30])
    result = LinearAlgebra.sum(vector_a, vector_b)

    assert_vector_values(result, [11, 22, 33])


def test_linear_algebra_sum_matrix_3x3_valid():
    matrix_a = Matrix(3, 3, [1, 2, 3, 4, 5, 6, 7, 8, 9])
    matrix_b = Matrix(3, 3, [9, 8, 7, 6, 5, 4, 3, 2, 1])
    result = LinearAlgebra.sum(matrix_a, matrix_b)

    assert_matrix_values(result, [[10, 10, 10], [10, 10, 10], [10, 10, 10]])


def test_linear_algebra_sum_invalid_dimension():
    matrix_a = Matrix(2, 2, [1, 2, 3, 4])
    matrix_b = Matrix(1, 2, [5, 6])

    assert_raises(
        ValueError,
        LinearAlgebra.sum,
        matrix_a,
        matrix_b,
        message_contains="mesma dimensao",
    )


def test_linear_algebra_sum_invalid_different_orders():
    matrix_a = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    matrix_b = Matrix(3, 2, [7, 8, 9, 10, 11, 12])

    assert_raises(
        ValueError,
        LinearAlgebra.sum,
        matrix_a,
        matrix_b,
        message_contains="mesma dimensao",
    )


def test_linear_algebra_sum_invalid_types():
    matrix = Matrix(2, 2, [1, 2, 3, 4])
    vector = Vector(2, [5, 6])

    assert_raises(
        TypeError,
        LinearAlgebra.sum,
        matrix,
        vector,
        message_contains="Matrix com Matrix ou Vector com Vector",
    )


def test_linear_algebra_times_scalar_matrix():
    matrix = Matrix(2, 2, [1, 2, 3, 4])
    result = LinearAlgebra.times(2, matrix)

    assert_matrix_values(result, [[2, 4], [6, 8]])


def test_linear_algebra_times_scalar_vector():
    vector = Vector(3, [1, 2, 3])
    result = LinearAlgebra.times(3, vector)

    assert_vector_values(result, [3, 6, 9])


def test_linear_algebra_times_matrix_scalar():
    matrix = Matrix(2, 2, [1, 2, 3, 4])
    result = LinearAlgebra.times(matrix, 2)

    assert_matrix_values(result, [[2, 4], [6, 8]])


def test_linear_algebra_times_vector_scalar():
    vector = Vector(3, [1, 2, 3])
    result = LinearAlgebra.times(vector, 3)

    assert_vector_values(result, [3, 6, 9])


def test_linear_algebra_times_matrix_matrix_elementwise():
    matrix_a = Matrix(2, 2, [1, 2, 3, 4])
    matrix_b = Matrix(2, 2, [2, 3, 4, 5])
    result = LinearAlgebra.times(matrix_a, matrix_b)

    assert_matrix_values(result, [[2, 6], [12, 20]])


def test_linear_algebra_times_matrix_matrix_3x3_elementwise():
    matrix_a = Matrix(3, 3, [1, 2, 3, 4, 5, 6, 7, 8, 9])
    matrix_b = Matrix(3, 3, [9, 8, 7, 6, 5, 4, 3, 2, 1])
    result = LinearAlgebra.times(matrix_a, matrix_b)

    assert_matrix_values(result, [[9, 16, 21], [24, 25, 24], [21, 16, 9]])


def test_linear_algebra_times_vector_vector_elementwise():
    vector_a = Vector(3, [1, 2, 3])
    vector_b = Vector(3, [4, 5, 6])
    result = LinearAlgebra.times(vector_a, vector_b)

    assert_vector_values(result, [4, 10, 18])


def test_linear_algebra_times_invalid_dimension():
    matrix_a = Matrix(2, 2, [1, 2, 3, 4])
    matrix_b = Matrix(1, 2, [5, 6])

    assert_raises(
        ValueError,
        LinearAlgebra.times,
        matrix_a,
        matrix_b,
        message_contains="mesma dimensao",
    )


def test_linear_algebra_times_invalid_different_orders():
    matrix_a = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    matrix_b = Matrix(3, 2, [7, 8, 9, 10, 11, 12])

    assert_raises(
        ValueError,
        LinearAlgebra.times,
        matrix_a,
        matrix_b,
        message_contains="mesma dimensao",
    )


def test_linear_algebra_dot_valid():
    matrix_a = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    matrix_b = Matrix(3, 2, [7, 8, 9, 10, 11, 12])
    result = LinearAlgebra.dot(matrix_a, matrix_b)

    assert_matrix_values(result, [[58, 64], [139, 154]])


def test_linear_algebra_dot_2x3_by_3x4_valid():
    matrix_a = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    matrix_b = Matrix(3, 4, [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
    result = LinearAlgebra.dot(matrix_a, matrix_b)

    assert_matrix_values(result, [[74, 80, 86, 92], [173, 188, 203, 218]])


def test_linear_algebra_dot_invalid_dimension():
    matrix_a = Matrix(2, 2, [1, 2, 3, 4])
    matrix_b = Matrix(3, 1, [5, 6, 7])

    assert_raises(
        ValueError,
        LinearAlgebra.dot,
        matrix_a,
        matrix_b,
        message_contains="numero de colunas da primeira matriz",
    )


def test_linear_algebra_dot_invalid_different_dimensions():
    matrix_a = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    matrix_b = Matrix(2, 2, [7, 8, 9, 10])

    assert_raises(
        ValueError,
        LinearAlgebra.dot,
        matrix_a,
        matrix_b,
        message_contains="numero de colunas da primeira matriz",
    )


def test_linear_algebra_gauss_simple_matrix():
    matrix = Matrix(2, 2, [2, 1, 4, 3])
    result = LinearAlgebra.gauss(matrix)

    assert_matrix_values(result, [[4, 3], [0, -0.5]])
    assert_matrix_values(matrix, [[2, 1], [4, 3]])


def test_linear_algebra_gauss_3x3_matrix():
    matrix = Matrix(3, 3, [2, 1, -1, -3, -1, 2, -2, 1, 2])
    result = LinearAlgebra.gauss(matrix)

    assert_matrix_values(result, [[-3, -1, 2], [0, 5 / 3, 2 / 3], [0, 0, 0.2]])
    assert_matrix_values(matrix, [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])


def test_linear_algebra_gauss_with_zero_initial_pivot():
    matrix = Matrix(2, 2, [0, 2, 1, 3])
    result = LinearAlgebra.gauss(matrix)

    assert_matrix_values(result, [[1, 3], [0, 2]])


def test_linear_algebra_solve_unique_solution():
    augmented = Matrix(2, 3, [1, 1, 3, 2, -1, 0])
    result = LinearAlgebra.solve(augmented)

    assert_matrix_values(result, [[1, 0, 1], [0, 1, 2]])
    assert_matrix_values(augmented, [[1, 1, 3], [2, -1, 0]])


def test_linear_algebra_solve_3x4_unique_solution():
    augmented = Matrix(3, 4, [1, 1, 1, 6, 2, -1, 1, 3, 1, 2, -1, 2])
    result = LinearAlgebra.solve(augmented)

    assert_matrix_values(result, [[1, 0, 0, 1], [0, 1, 0, 2], [0, 0, 1, 3]])
    assert_matrix_values(augmented, [[1, 1, 1, 6], [2, -1, 1, 3], [1, 2, -1, 2]])


def test_linear_algebra_solve_invalid_system():
    invalid_augmented = Matrix(2, 1, [1, 2])

    assert_raises(
        ValueError,
        LinearAlgebra.solve,
        invalid_augmented,
        message_contains="pelo menos duas colunas",
    )


def test_linear_algebra_solve_impossible_system():
    augmented = Matrix(2, 3, [1, 1, 2, 2, 2, 5])

    assert_raises(
        ValueError,
        LinearAlgebra.solve,
        augmented,
        message_contains="impossivel",
    )


def test_linear_algebra_solve_impossible_3x4_system():
    augmented = Matrix(3, 4, [1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 3])

    assert_raises(
        ValueError,
        LinearAlgebra.solve,
        augmented,
        message_contains="impossivel",
    )


def test_linear_algebra_solve_dependent_system():
    augmented = Matrix(2, 3, [1, 1, 2, 2, 2, 4])

    assert_raises(
        ValueError,
        LinearAlgebra.solve,
        augmented,
        message_contains="multiplas solucoes",
    )


def test_linear_algebra_solve_dependent_3x4_system():
    augmented = Matrix(3, 4, [1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3])

    assert_raises(
        ValueError,
        LinearAlgebra.solve,
        augmented,
        message_contains="multiplas solucoes",
    )


def test_linear_algebra_gauss_jordan():
    matrix = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    result = LinearAlgebra.gauss_jordan(matrix)

    assert_matrix_values(result, [[1, 0, -1], [0, 1, 2]])


def test_linear_algebra_create_identity():
    result = LinearAlgebra.create_identity(3)

    assert_matrix_values(result, [[1, 0, 0], [0, 1, 0], [0, 0, 1]])


def test_linear_algebra_augment_matrix():
    matrix = Matrix(2, 2, [1, 2, 3, 4])
    vector = Vector(2, [5, 6])
    result = LinearAlgebra.augment_matrix(matrix, vector)

    assert_matrix_values(result, [[1, 2, 5], [3, 4, 6]])


def test_linear_algebra_solve_system():
    matrix = Matrix(2, 2, [1, 1, 2, -1])
    vector = Vector(2, [3, 0])
    result = LinearAlgebra.solve_system(matrix, vector)

    assert_matrix_values(result, [[1, 0, 1], [0, 1, 2]])


def test_linear_algebra_validate_same_matrix_order():
    matrix_a = Matrix(2, 2, [1, 2, 3, 4])
    matrix_b = Matrix(2, 2, [5, 6, 7, 8])

    LinearAlgebra.validate_same_matrix_order(matrix_a, matrix_b)


def test_linear_algebra_validate_same_vector_size():
    vector_a = Vector(3, [1, 2, 3])
    vector_b = Vector(3, [4, 5, 6])

    LinearAlgebra.validate_same_vector_size(vector_a, vector_b)


def test_linear_algebra_validate_matrix_vector_dimensions():
    matrix = Matrix(2, 2, [1, 2, 3, 4])
    vector = Vector(2, [5, 6])

    LinearAlgebra.validate_matrix_vector_dimensions(matrix, vector)


def test_linear_algebra_augment_matrix_invalid_dimension():
    matrix = Matrix(2, 3, [1, 2, 3, 4, 5, 6])
    vector = Vector(3, [7, 8, 9])

    assert_raises(
        ValueError,
        LinearAlgebra.augment_matrix,
        matrix,
        vector,
        message_contains="numero de linhas da matriz",
    )


def run_all_tests():
    tests = [
        test_matrix_creation_valid,
        test_matrix_creation_invalid_size,
        test_matrix_creation_invalid_number_of_elements,
        test_matrix_get_valid,
        test_matrix_get_invalid,
        test_matrix_set_valid,
        test_matrix_set_invalid,
        test_matrix_copy_creates_independent_object,
        test_matrix_wrappers,
        test_matrix_row_operations,
        test_matrix_gauss_jordan_elimination_wrapper,
        test_vector_creation_valid,
        test_vector_creation_invalid_dimension,
        test_vector_creation_invalid_number_of_elements,
        test_vector_get_valid,
        test_vector_get_invalid,
        test_vector_set_valid,
        test_vector_set_invalid,
        test_vector_copy_creates_independent_object,
        test_vector_wrappers_and_dot,
        test_vector_to_column_matrix,
        test_linear_algebra_transpose_matrix,
        test_linear_algebra_transpose_matrix_3x2,
        test_linear_algebra_transpose_vector,
        test_linear_algebra_sum_matrix_valid,
        test_linear_algebra_sum_vector_valid,
        test_linear_algebra_sum_matrix_3x3_valid,
        test_linear_algebra_sum_invalid_dimension,
        test_linear_algebra_sum_invalid_different_orders,
        test_linear_algebra_sum_invalid_types,
        test_linear_algebra_times_scalar_matrix,
        test_linear_algebra_times_scalar_vector,
        test_linear_algebra_times_matrix_scalar,
        test_linear_algebra_times_vector_scalar,
        test_linear_algebra_times_matrix_matrix_elementwise,
        test_linear_algebra_times_matrix_matrix_3x3_elementwise,
        test_linear_algebra_times_vector_vector_elementwise,
        test_linear_algebra_times_invalid_dimension,
        test_linear_algebra_times_invalid_different_orders,
        test_linear_algebra_dot_valid,
        test_linear_algebra_dot_2x3_by_3x4_valid,
        test_linear_algebra_dot_invalid_dimension,
        test_linear_algebra_dot_invalid_different_dimensions,
        test_linear_algebra_gauss_simple_matrix,
        test_linear_algebra_gauss_3x3_matrix,
        test_linear_algebra_gauss_with_zero_initial_pivot,
        test_linear_algebra_gauss_jordan,
        test_linear_algebra_solve_unique_solution,
        test_linear_algebra_solve_3x4_unique_solution,
        test_linear_algebra_solve_invalid_system,
        test_linear_algebra_solve_impossible_system,
        test_linear_algebra_solve_impossible_3x4_system,
        test_linear_algebra_solve_dependent_system,
        test_linear_algebra_solve_dependent_3x4_system,
        test_linear_algebra_create_identity,
        test_linear_algebra_augment_matrix,
        test_linear_algebra_solve_system,
        test_linear_algebra_validate_same_matrix_order,
        test_linear_algebra_validate_same_vector_size,
        test_linear_algebra_validate_matrix_vector_dimensions,
        test_linear_algebra_augment_matrix_invalid_dimension,
    ]

    for test in tests:
        test()


if __name__ == "__main__":
    run_all_tests()
    print("Testes funcionais executados com sucesso.")


## Demonstração do projeto
Execute a célula abaixo para rodar o `main.py` sem alterar o código acima.

In [ ]:
# Execução da demonstração principal
if "main" in globals():
    main()
else:
    print("Função main não encontrada.")


## Execução dos testes
A célula abaixo executa automaticamente todas as funções de teste que começam com `test_`.

In [ ]:
test_functions = sorted(
    name for name, obj in globals().items()
    if name.startswith("test_") and callable(obj)
)

executed = 0
for test_name in test_functions:
    globals()[test_name]()
    executed += 1

print(f"✅ Todos os testes passaram com sucesso. Total executado: {executed}")


## Encerramento

Este notebook foi preparado para facilitar a **apresentação**, mantendo a **estrutura original do projeto** e organizando o conteúdo de forma mais limpa no Colab.
